# Stage 8 — Noise Robustness

In this stage we evaluate how robust our models are to **additive noise** and **compression artifacts**.

We focus on:

- the 4 fine-tuned transformers from Stage 6, and
- (optionally) their corresponding **fusion models** from Stage 7.

All experiments use the **same speaker-disjoint split** (`results/split.json`) and the same decision thresholds tuned on validation F1 in earlier stages.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config import OUTPUTS_DIR

noise_path = OUTPUTS_DIR / "noise_robustness_results.json"

with open(noise_path) as f:
    noise_results = json.load(f)

list(noise_results.keys())

In [ ]:
def to_df(model_key: str) -> pd.DataFrame:
    """
    Convert noise_results[model_key] to a tidy DataFrame.
    Expect each entry to have at least:
      - snr_db
      - noise_type
      - auc_roc
      - f1
    """
    rows = noise_results[model_key]
    return pd.DataFrame(rows)

# Example: inspect HuBERT
df_hubert = to_df("hubert_base_ls960")
df_hubert.head()

In [ ]:
models = ["hubert_base_ls960", "wav2vec2", "wavlm", "whisper_tiny"]
colors = ["C0", "C1", "C2", "C3"]

plt.figure(figsize=(8, 5))
for m, c in zip(models, colors):
    df = to_df(m)
    # aggregate over noise types if you have multiple types
    df_group = df.groupby("snr_db")["auc_roc"].mean().reset_index()
    plt.plot(df_group["snr_db"], df_group["auc_roc"], "o-", label=m, color=c)

plt.gca().invert_xaxis()  # optional: show clean→0 dB left→right
plt.xlabel("SNR (dB)")
plt.ylabel("Test AUC-ROC")
plt.ylim(0, 1.05)
plt.title("Noise robustness — AUC vs SNR")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
for m, c in zip(models, colors):
    df = to_df(m)
    df_group = df.groupby("snr_db")["f1"].mean().reset_index()
    plt.plot(df_group["snr_db"], df_group["f1"], "o-", label=m, color=c)

plt.gca().invert_xaxis()
plt.xlabel("SNR (dB)")
plt.ylabel("Test F1")
plt.ylim(0, 1.05)
plt.title("Noise robustness — F1 vs SNR")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
def plot_per_noise_type(model_key: str):
    df = to_df(model_key)
    noise_types = sorted(df["noise_type"].unique())
    plt.figure(figsize=(8, 5))
    for nt in noise_types:
        sub = df[df["noise_type"] == nt]
        sub_group = sub.groupby("snr_db")["auc_roc"].mean().reset_index()
        plt.plot(sub_group["snr_db"], sub_group["auc_roc"], "o-", label=nt)
    plt.gca().invert_xaxis()
    plt.xlabel("SNR (dB)")
    plt.ylabel("Test AUC-ROC")
    plt.ylim(0, 1.05)
    plt.title(f"Noise robustness by type — {model_key}")
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_per_noise_type("wavlm")

In [ ]:
# Example extension if noise_results has a 'fusion' section:
has_fusion = "fusion" in noise_results

if has_fusion:
    plt.figure(figsize=(8, 5))
    for m, c in zip(models, colors):
        base_df = pd.DataFrame(noise_results["base"][m])
        fus_df  = pd.DataFrame(noise_results["fusion"][m])
        base_group = base_df.groupby("snr_db")["auc_roc"].mean().reset_index()
        fus_group  = fus_df.groupby("snr_db")["auc_roc"].mean().reset_index()
        plt.plot(base_group["snr_db"], base_group["auc_roc"], "o-", label=f"{m} base", color=c)
        plt.plot(fus_group["snr_db"], fus_group["auc_roc"], "--", label=f"{m} fusion", color=c)

    plt.gca().invert_xaxis()
    plt.xlabel("SNR (dB)")
    plt.ylabel("Test AUC-ROC")
    plt.ylim(0, 1.05)
    plt.title("Noise robustness — Base vs Fusion (AUC)")
    plt.legend()
    plt.tight_layout()
    plt.show()

## Interpretation

- All models use the same **speaker-disjoint train/val/test split** and the same
  decision thresholds tuned on **validation F1** in the clean condition.

- The AUC vs SNR and F1 vs SNR plots show how each architecture degrades as
  noise level increases (20 dB → 10 dB → 5 dB → 0 dB, plus clean).

- In the report, we highlight:
  - which transformer is most robust under additive noise and compression,
  - whether end-to-end fusion (Stage 7) improves or hurts robustness when
    evaluated under the same thresholding and split.